## BME 230 Intro modeling RNA.ipynb


### Problem 1

**Briefly answer each question below:**

**(a)** What is the role of the unique molecular identifier (UMI)?
The UMI is used to uniquely tag individual RNA molecules during library preparation to eliminate amplification bias. By identifying and collapsing duplicate reads with the same UMI, researchers can more accurately quantify the number of unique RNA transcripts in a sample.

**(b)** What is the ‘library’ produced by single-cell sequencing methods?
The library refers to the collection of DNA fragments derived from the RNA molecules of the cells. These fragments include adapters, barcodes, and UMIs, allowing them to be sequenced and assigned to their respective cells and transcripts.


**(c)** What does 3’ capture refer to (as in 3’ sequencing methods)?
3’ capture refers to the sequencing of the 3' end of RNA molecules, which includes the poly(A) tail and adjacent sequence regions. This method focuses on capturing transcript ends to identify gene expression levels without sequencing full-length transcripts.

**(d)** For 3’ 10X sequencing, write the order in which the events below are executed for generating the final, sequenced library:

     i) cDNA fragmentation and size selection  
     ii) Cell capture and lysis  
     iii) transcript capture and barcoding  
     iv)Reverse transcription and amplification  
     v) Addition of sample index/label (sample index PCR)  
     vi)Single- or paired-end 
     
     The correct order is:
ii → iii → iv → i → v → vi

Cell capture and lysis
Transcript capture and barcoding
Reverse transcription and amplification
cDNA fragmentation and size selection
Addition of sample index/label (sample index PCR)
Single- or paired-end sequencing


### Problem 2 

Consider the following gene expression matrix \( G \), containing the expression level of four genes across three cells. The \( i \)-th row of \( G \) represents the expression level of all genes in the \( i \)-th cell, and the \( j \)-th column represents the expression level of the \( j \)-th gene across all cells.

$$
G =
\begin{bmatrix}
1 & 2 & 3 & 3 \\\
3 & 1 & 9 & 4 \\\
1 & 4 & 3 & 5
\end{bmatrix}
$$

**(a)**  Which cell has the lowest expression level of gene 3 (index starts from 1), Which gene is most highly expressed in cell 1?   


Consider three valid distance functions between a pair of vectors **x** and **y**, the L1 distance, L2 distance, and cosine similarity c :

$$
L_1(\mathbf{x}, \mathbf{y}) := \|\mathbf{x} - \mathbf{y}\|_1 = \sum_{i=1}^{n} |x_i - y_i|,
$$

$$
L_2(\mathbf{x}, \mathbf{y}) := \|\mathbf{x} - \mathbf{y}\|_2 = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2},
$$

$$
c(\mathbf{x}, \mathbf{y}) := \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\|_2 \|\mathbf{y}\|_2} = \frac{\sum_{i=1}^{n} x_i y_i}{\sqrt{\sum_{i=1}^{n} x_i^2} \sqrt{\sum_{i=1}^{n} y_i^2}}.
$$

Note that the cosine similarity is simply a measure of the cosine of the angle between two vectors. 

**(b)** For each distance function above, construct a distance matrix \( D \), where \( D{ij} \) denotes the distance between the ith and jth row cell of G.

**(c)** For each distance function, which two cells in G are most similar to one another?  For L1 and L2 distance, this corresponds to a pair of row vectors with the smallest distance. For cosine similarity, this corresponds to a pair of row vectors with the smallest angle.


## Load data for analysis

### Problem 3

In [6]:
import scipy.io as sio
import pandas as pd
import requests
from tqdm import tnrange, tqdm_notebook
import matplotlib.pyplot as plt


Cells are 10x sequenced neurons from the mouse hypothalamus (Kim et al. 2019)

Paper link : https://www.sciencedirect.com/science/article/pii/S0092867419310712


count matrix is 41,580 cells by 1,999 genes. We will use the full dataset to fit these models.

For each cell, gene counts were normalized to have the same number of total counts (usually 1e5 or 1e6), with cell-gene counts thus scaled accordingly.

Counts were then log-normalized, using the log(1+x), where x is each cell's gene count. The 1 accounts for 0 count genes.

The ~2000 genes were selected for those that displayed large variance in expression amongst the cells ('highly variable genes').




In [ ]:
import requests
from tqdm import tqdm 

def download_file(doi, ext):
    url = 'https://api.datacite.org/dois/' + doi + '/media'
    r = requests.get(url).json()
    if 'data' not in r or len(r['data']) == 0:
        print("File information not found for DOI:", doi)
        return

    netcdf_url = r['data'][0]['attributes']['url']
    r = requests.get(netcdf_url, stream=True)
    fname = doi.split('/')[-1] + ext

    if r.status_code == 403:
        print("File Unavailable")
        return
    if 'content-length' not in r.headers:
        print("Did not get file")
        return

    with open(fname, 'wb') as f:
        total_length = int(r.headers.get('content-length'))
        pbar = tqdm(total=total_length, unit="B", unit_scale=True, desc=f"Downloading {fname}")
        for chunk in r.iter_content(chunk_size=1024):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))
        pbar.close()

    return fname

# Test the function with the DOIs
download_file('10.22002/D1.2072', '.gz')
download_file('10.22002/D1.8969', '.gz')
download_file('10.22002/D1.2066', '.gz')


In [81]:
!gunzip *.gz

In [82]:
!mv D1.2072 tenx.mtx # gene expression 
!mv D1.8969 tenx_obs.csv # meta data
!mv D1.2066 var.csv # meta_gene 

In [42]:
#Get gene count matrix (.mtx) # use scipy library 
#Get meta data(.csv) 
#Get meta_genedata(.csv) 

Find x ~ y and y ~ x regression model coefficients for a pair of genes and plot x versus y in each case.

gene 1: 'Gm31363', gene2 : 'Npbwr1'

By regressing y (gene 2) on x (gene 1) written as y ~ x, or vice versa, we are modeling how gene 2's expression changes as a function of gene 1. Specifically, the $\beta$ parameter from the fit $y = \alpha + \beta x$ represents the change in the value of dependent variable (y here) corresponding to unit change in the value of independent variable (x here). 

**(a)** Calculate the $R^2$ and the Pearson correlation coefficient for the gene pair, given each regression model.

The  $R^2$ coefficient of determination is defined as 1 - (sum of squares of the residuals)/(sum of total squares). Here the numerator represents deviation from the model predictions, and the denominator represents the variance of the given dataset (observations). $R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum_i (y_i - y_{pred})^2}{\sum_i (y_i - \bar{y})^2} $ (where $\bar{y}$ is the mean). This measures the proportion of the variation in y that is predictable from x.



In [76]:
# your code below 
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# Load data
counts = pd.read_csv('tenx.mtx', delim_whitespace=True, header=None)  # Expression matrix
metadata = pd.read_csv('tenx_obs.csv')  # Metadata
genes = pd.read_csv('var.csv')  # Gene metadata

# Get the indices of the genes of interest
gene_1 = 'Gm31363'
gene_2 = 'Npbwr1'

gene_1_index = genes[genes['gene_name'] == gene_1].index[0]
gene_2_index = genes[genes['gene_name'] == gene_2].index[0]

# Extract gene expression values
gene_1_expression = counts.iloc[:, gene_1_index]
gene_2_expression = counts.iloc[:, gene_2_index]

# Prepare DataFrame
df = pd.DataFrame({
    'gene_1': gene_1_expression,
    'gene_2': gene_2_expression
})

# (a) y ~ x: Regress gene_2 on gene_1
model_y_on_x = LinearRegression().fit(df[['gene_1']], df['gene_2'])
alpha_y_on_x = model_y_on_x.intercept_
beta_y_on_x = model_y_on_x.coef_[0]
y_pred_from_x = model_y_on_x.predict(df[['gene_1']])
r2_y_on_x = r2_score(df['gene_2'], y_pred_from_x)
pearson_y_on_x, _ = pearsonr(df['gene_1'], df['gene_2'])

# (b) x ~ y: Regress gene_1 on gene_2
model_x_on_y = LinearRegression().fit(df[['gene_2']], df['gene_1'])
alpha_x_on_y = model_x_on_y.intercept_
beta_x_on_y = model_x_on_y.coef_[0]
x_pred_from_y = model_x_on_y.predict(df[['gene_2']])
r2_x_on_y = r2_score(df['gene_1'], x_pred_from_y)
pearson_x_on_y, _ = pearsonr(df['gene_2'], df['gene_1'])

# Plot results
plt.figure(figsize=(12, 6))

# Plot y ~ x
plt.subplot(1, 2, 1)
plt.scatter(df['gene_1'], df['gene_2'], label='Data points', alpha=0.6)
plt.plot(df['gene_1'], y_pred_from_x, color='red', label=f'Regression Line: y ~ x')
plt.xlabel('Gene 1 Expression')
plt.ylabel('Gene 2 Expression')
plt.title(f'y ~ x | β={beta_y_on_x:.3f}, R²={r2_y_on_x:.3f}, Pearson r={pearson_y_on_x:.3f}')
plt.legend()

# Plot x ~ y
plt.subplot(1, 2, 2)
plt.scatter(df['gene_2'], df['gene_1'], label='Data points', alpha=0.6)
plt.plot(df['gene_2'], x_pred_from_y, color='blue', label=f'Regression Line: x ~ y')
plt.xlabel('Gene 2 Expression')
plt.ylabel('Gene 1 Expression')
plt.title(f'x ~ y | β={beta_x_on_y:.3f}, R²={r2_x_on_y:.3f}, Pearson r={pearson_x_on_y:.3f}')
plt.legend()

plt.tight_layout()
plt.show()

# Print coefficients and R²
print(f"Regression y ~ x: α = {alpha_y_on_x:.3f}, β = {beta_y_on_x:.3f}, R² = {r2_y_on_x:.3f}, Pearson r = {pearson_y_on_x:.3f}")
print(f"Regression x ~ y: α = {alpha_x_on_y:.3f}, β = {beta_x_on_y:.3f}, R² = {r2_x_on_y:.3f}, Pearson r = {pearson_x_on_y:.3f}")


**(b)** Partial correlation is a measure of association between two variables, after controlling for the effect of a third random variable. Perform linear regression to model the expression of each gene as a function of cell sex. Report the coefficient and intercept for each model.

Fit linear regression models to predict:

1. Gene 1 expression
2. Gene 2 expression

based on the sex of cells, represented as a binary variable (0 for Male and 1 for Female). Use the entire count matrix, which includes data from both sexes and all cell types. Before fitting the models, convert the 'M' and 'F' labels for sex into binary values (0 or 1) for use in the regression analysis. Find and report the partial correlation between the genes in pair. 


In [75]:
# your code below 
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr

# Load data
# Ensure the file paths and delimiters match your data format.
# Adjust column names if the files have headers.
counts = pd.read_csv('tenx.mtx', delim_whitespace=True, header=None)  # Gene expression matrix
metadata = pd.read_csv('tenx_obs.csv')  # Metadata with cell information
genes = pd.read_csv('var.csv')  # Gene metadata

# Map 'M' and 'F' to binary values: 0 (Male) and 1 (Female)
metadata['sex_binary'] = metadata['sex'].map({'M': 0, 'F': 1})

# Get the indices of the genes of interest
gene_1 = 'Gm31363'
gene_2 = 'Npbwr1'

gene_1_index = genes[genes['gene_name'] == gene_1].index[0]
gene_2_index = genes[genes['gene_name'] == gene_2].index[0]

# Extract the expression levels for the two genes
gene_1_expression = counts.iloc[:, gene_1_index]
gene_2_expression = counts.iloc[:, gene_2_index]

# Add the gene expressions and sex_binary to a DataFrame for regression
df = pd.DataFrame({
    'gene_1': gene_1_expression,
    'gene_2': gene_2_expression,
    'sex': metadata['sex_binary']
})

# Fit linear regression models
# Gene 1 expression as a function of sex
model_gene_1 = LinearRegression().fit(df[['sex']], df['gene_1'])
coef_gene_1 = model_gene_1.coef_[0]
intercept_gene_1 = model_gene_1.intercept_

# Gene 2 expression as a function of sex
model_gene_2 = LinearRegression().fit(df[['sex']], df['gene_2'])
coef_gene_2 = model_gene_2.coef_[0]
intercept_gene_2 = model_gene_2.intercept_

# Compute residuals
df['residual_gene_1'] = df['gene_1'] - model_gene_1.predict(df[['sex']])
df['residual_gene_2'] = df['gene_2'] - model_gene_2.predict(df[['sex']])

# Compute partial correlation
partial_corr, _ = pearsonr(df['residual_gene_1'], df['residual_gene_2'])

# Report results
print(f"Gene 1 Regression: Coefficient = {coef_gene_1}, Intercept = {intercept_gene_1}")
print(f"Gene 2 Regression: Coefficient = {coef_gene_2}, Intercept = {intercept_gene_2}")
print(f"Partial Correlation between Gene 1 and Gene 2: {partial_corr}")


## Problem 4



The expression levels of two genes, X1 and X2, are modeled as follows:

$$
X_1 = Z + \epsilon_1, \quad X_2 = Z + \epsilon_2,
$$

where \(Z\) is a negative binomial random variable with:

$$
\mathbb{E}[Z] = \frac{n(1-p)}{p}, \quad \text{Var}(Z) = \frac{n(1-p)}{p^2}.
$$

The terms e1 and e2 are independent, identically distributed Poisson random variables with:

$$
\mathbb{E}[\epsilon_1] = \mathbb{E}[\epsilon_2] = \lambda,
$$

and both e1 and e2 are independent of Z.

For this problem, the parameters are set as follows:

$$
\lambda = 40, \quad n = 1, \quad p = 0.3.
$$


(a). Write a script to generate 100,000 pairs of X1, X2 based on the given model and compute the Pearson correlation between X1 and X2 in the full dataset.




In [78]:
import numpy as np
from scipy.stats import nbinom, poisson, pearsonr

# Parameters
lambda_param = 40
n = 1
p = 0.3
num_samples = 100000

# Step 1: Simulate Z (Negative Binomial distribution)
mean_z = n * (1 - p) / p
var_z = n * (1 - p) / (p ** 2)
k = n
r = (1 - p) / p
Z = nbinom.rvs(k, p, size=num_samples)

# Step 2: Simulate epsilon_1 and epsilon_2 (Poisson distribution)
epsilon_1 = poisson.rvs(lambda_param, size=num_samples)
epsilon_2 = poisson.rvs(lambda_param, size=num_samples)

# Step 3: Compute X1 and X2
X1 = Z + epsilon_1
X2 = Z + epsilon_2

# Step 4: Compute Pearson correlation
correlation, _ = pearsonr(X1, X2)

print(f"Pearson correlation between X1 and X2: {correlation:.4f}")

(b). Apply a filtering condition where only data points satisfying X1 > X2 are retained. Compute the correlation between X1 and X2 in the filtered dataset and create a scatter plot comparing the original and filtered data (use different colors for clarity).




In [ ]:
# your code below 
import numpy as np
from scipy.stats import nbinom, poisson, pearsonr
import matplotlib.pyplot as plt

# Parameters
lambda_param = 40
n = 1
p = 0.3
num_samples = 100000

# Step 1: Simulate Z (Negative Binomial distribution)
mean_z = n * (1 - p) / p
var_z = n * (1 - p) / (p ** 2)
k = n
r = (1 - p) / p
Z = nbinom.rvs(k, p, size=num_samples)

# Step 2: Simulate epsilon_1 and epsilon_2 (Poisson distribution)
epsilon_1 = poisson.rvs(lambda_param, size=num_samples)
epsilon_2 = poisson.rvs(lambda_param, size=num_samples)

# Step 3: Compute X1 and X2
X1 = Z + epsilon_1
X2 = Z + epsilon_2

# Step 4: Compute Pearson correlation for full dataset
correlation_full, _ = pearsonr(X1, X2)
print(f"Pearson correlation between X1 and X2 (full dataset): {correlation_full:.4f}")

# Step 5: Apply filtering condition (X1 > X2)
filtered_indices = X1 > X2
X1_filtered = X1[filtered_indices]
X2_filtered = X2[filtered_indices]

# Compute Pearson correlation for filtered dataset
correlation_filtered, _ = pearsonr(X1_filtered, X2_filtered)
print(f"Pearson correlation between X1 and X2 (filtered dataset): {correlation_filtered:.4f}")

# Step 6: Create scatter plots
plt.figure(figsize=(12, 6))

# Scatter plot for full dataset
plt.subplot(1, 2, 1)
plt.scatter(X1, X2, alpha=0.2, label="Full dataset", color="blue")
plt.title("Scatter Plot: Full Dataset")
plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()

# Scatter plot for filtered dataset
plt.subplot(1, 2, 2)
plt.scatter(X1_filtered, X2_filtered, alpha=0.2, label="Filtered dataset (X1 > X2)", color="orange")
plt.title("Scatter Plot: Filtered Dataset")
plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()

# Show plots
plt.tight_layout()
plt.show()

(c). Repeat the filtering process with the condition $$|X_1 - X_2| > 50$$ Compute the correlation for this filtered dataset and plot the scatter plot of the resulting points alongside the original dataset. Compare and describe the results with those obtained in parts (a) and (b).

In [79]:
# your code below 
import numpy as np
from scipy.stats import nbinom, poisson, pearsonr
import matplotlib.pyplot as plt

# Parameters
lambda_param = 40
n = 1
p = 0.3
num_samples = 100000

# Step 1: Simulate Z (Negative Binomial distribution)
mean_z = n * (1 - p) / p
var_z = n * (1 - p) / (p ** 2)
k = n
r = (1 - p) / p
Z = nbinom.rvs(k, p, size=num_samples)

# Step 2: Simulate epsilon_1 and epsilon_2 (Poisson distribution)
epsilon_1 = poisson.rvs(lambda_param, size=num_samples)
epsilon_2 = poisson.rvs(lambda_param, size=num_samples)

# Step 3: Compute X1 and X2
X1 = Z + epsilon_1
X2 = Z + epsilon_2

# Step 4: Compute Pearson correlation for full dataset
correlation_full, _ = pearsonr(X1, X2)
print(f"Pearson correlation between X1 and X2 (full dataset): {correlation_full:.4f}")

# Step 5: Apply filtering condition (X1 > X2)
filtered_indices_x1_gt_x2 = X1 > X2
X1_filtered_x1_gt_x2 = X1[filtered_indices_x1_gt_x2]
X2_filtered_x1_gt_x2 = X2[filtered_indices_x1_gt_x2]

# Compute Pearson correlation for filtered dataset (X1 > X2)
correlation_filtered_x1_gt_x2, _ = pearsonr(X1_filtered_x1_gt_x2, X2_filtered_x1_gt_x2)
print(f"Pearson correlation between X1 and X2 (filtered dataset, X1 > X2): {correlation_filtered_x1_gt_x2:.4f}")

# Step 6: Apply filtering condition (|X1 - X2| > 50)
filtered_indices_abs_diff = np.abs(X1 - X2) > 50
X1_filtered_abs_diff = X1[filtered_indices_abs_diff]
X2_filtered_abs_diff = X2[filtered_indices_abs_diff]

# Compute Pearson correlation for filtered dataset (|X1 - X2| > 50)
correlation_filtered_abs_diff, _ = pearsonr(X1_filtered_abs_diff, X2_filtered_abs_diff)
print(f"Pearson correlation between X1 and X2 (filtered dataset, |X1 - X2| > 50): {correlation_filtered_abs_diff:.4f}")

# Step 7: Create scatter plots
plt.figure(figsize=(18, 6))

# Scatter plot for full dataset
plt.subplot(1, 3, 1)
plt.scatter(X1, X2, alpha=0.2, label="Full dataset", color="blue")
plt.title("Scatter Plot: Full Dataset")
plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()

# Scatter plot for filtered dataset (X1 > X2)
plt.subplot(1, 3, 2)
plt.scatter(X1_filtered_x1_gt_x2, X2_filtered_x1_gt_x2, alpha=0.2, label="Filtered dataset (X1 > X2)", color="orange")
plt.title("Scatter Plot: Filtered Dataset (X1 > X2)")
plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()

# Scatter plot for filtered dataset (|X1 - X2| > 50)
plt.subplot(1, 3, 3)
plt.scatter(X1_filtered_abs_diff, X2_filtered_abs_diff, alpha=0.2, label="Filtered dataset (|X1 - X2| > 50)", color="green")
plt.title("Scatter Plot: Filtered Dataset (|X1 - X2| > 50)")
plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()

# Show plots
plt.tight_layout()
plt.show()
